# So sánh mức độ bền vững hai model Logistic Regression và Decision Tree khi bị tấn công

Các notebook trước tập trung riêng vào Logistic Regression — cả khi bị tấn công (`01_training_model_lr.ipynb`), khi đi tìm dấu hiệu nhận biết tấn công (`anwareness-attack.ipynb`), lẫn khi thử các biện pháp phòng thủ (`resolve-problems.ipynb`). Notebook này đặt câu hỏi rộng hơn: **giữa hai kiến trúc mô hình khác nhau — Logistic Regression (tuyến tính) và Decision Tree (phi tuyến, dựa trên luật chia nhánh) — mô hình nào "bền" hơn trước tấn công label flipping?**
 
Cả hai mô hình (`lr_trained`, `dt_trained`) đã được huấn luyện sẵn ở `src/models`, mỗi loại gồm nhiều mô hình con ứng với từng mức % nhãn bị đảo (0%, 5%, ..., 45%) — đây cũng chính là cặp mô hình đã dùng ở Case 3 của `anwareness-attack.ipynb` để đo tỉ lệ bất đồng (disagreement rate). Notebook này đi theo hướng bổ sung: thay vì đo hai mô hình *bất đồng với nhau* bao nhiêu, đo trực tiếp **accuracy thật của từng mô hình** trên cùng tập test sạch, để so sánh ai suy giảm nhanh hơn.

In [ ]:
import sys, os 
from pathlib import Path

project_root = Path.cwd().parent 

os.chdir(project_root)
sys.path.append(str(project_root))

In [ ]:
# Python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Project
from src.defend import cleaning
from src.preprocess import X_test, y_test, train_dfs
from src.models import lr_trained, dt_trained
from src.utils.config import config

## Chuẩn bị

Vì tấn công label flipping chỉ đảo nhãn (`attack`), không thay đổi đặc trưng (`X`), nên `X` có phân phối giống nhau ở mọi mức % flip — chỉ cần fit một `StandardScaler` duy nhất (ở đây fit trên `train_dfs["45"]["X"]`, mức đầu độc nặng nhất) rồi áp dụng cho `X_test`, tạo ra `X_test_scaled` dùng chung cho toàn bộ phép so sánh, không cần tính lại cho từng mức %.

In [ ]:
# Because the X data is the same for each poisoned level
# do not need to scale X_test by all cases
scaler = StandardScaler().fit(train_dfs["45"]["X"])
X_test_scaled = scaler.transform(X_test)

## So sánh hai models

### 1. Chuẩn bị

Lấy dải % flip từ `config["poisoned-level"]` (dạng `[start, stop, step]`, tương ứng 0 → 45, bước nhảy 5 → 10 mức % flip). Với mỗi mức, lấy đúng cặp mô hình `lr_trained[str(percent)]` và `dt_trained[str(percent)]` đã huấn luyện sẵn cho mức đó, dự đoán trên `X_test_scaled`, tính `accuracy_score` so với `y_test` thật, gom kết quả vào `acc_results_df` (2 cột `acc_lr`, `acc_dt` theo từng `percent`).

In [ ]:
acc_results = []

flip_percent = range(
    config["poisoned-level"][0],
    config["poisoned-level"][1],
    config["poisoned-level"][2]
)

for percent in flip_percent:
    y_lr_predict = lr_trained[str(percent)].predict(X_test_scaled)
    y_dt_predict = dt_trained[str(percent)].predict(X_test_scaled)

    acc_results.append({
        "percent": percent,
        "acc_lr": accuracy_score(y_test, y_lr_predict),
        "acc_dt": accuracy_score(y_test, y_dt_predict)
    })

acc_results_df = pd.DataFrame(acc_results)

### 2. Vẽ biểu đồ

**Kết quả:**
 
| % flip | Accuracy — LR | Accuracy — DT | Chênh lệch (LR − DT) |
|---|---|---|---|
| 0% | ≈ 0.96 | ≈ 0.94 | +0.02 |
| 5% | ≈ 0.91 | ≈ 0.87 | +0.04 |
| 10% | ≈ 0.88 | ≈ 0.85 | +0.03 |
| 15% | ≈ 0.83 | ≈ 0.78 | +0.05 |
| 20% | ≈ 0.81 | ≈ 0.73 | +0.08 |
| 25% | ≈ 0.75 | ≈ 0.68 | +0.07 |
| 30% | ≈ 0.70 | ≈ 0.65 | +0.05 |
| 35% | ≈ 0.68 | ≈ 0.60 | +0.08 |
| 40% | ≈ 0.60 | ≈ 0.59 | +0.01 |
| 45% | ≈ 0.58 | ≈ 0.545 | +0.035 |

Hai quan sát chính:
 
1. **Cả hai mô hình đều suy giảm accuracy gần như tuyến tính** theo % flip tăng dần — không có mô hình nào "chống chịu" hoàn toàn hay sụp đổ đột ngột ở một ngưỡng cụ thể. Điều này phù hợp với bản chất của label flipping: mức độ nhiễu tăng dần một cách liên tục, không có "điểm gãy" rõ rệt.
2. **Logistic Regression bền hơn Decision Tree ở hầu hết các mức %** — đường LR nằm trên đường DT xuyên suốt từ 5% đến 45% (trừ điểm 40% gần như ngang nhau). Chênh lệch dao động khoảng 0.01 – 0.08 điểm accuracy, rõ nhất ở vùng 15–35% (chênh ~0.05–0.08). Điều này khá hợp lý về mặt trực giác: Decision Tree có xu hướng chia nhánh dựa trên từng điểm dữ liệu cụ thể (kể cả điểm nhiễu), dễ "học thuộc" các mẫu bị đảo nhãn thành các luật chia nhánh sai; trong khi Logistic Regression học một ranh giới quyết định toàn cục (một siêu phẳng), nên chịu ảnh hưởng của nhiễu nhãn theo kiểu trung bình hóa hơn là bị từng điểm nhiễu "kéo lệch" cục bộ.
> **Lưu ý:** so sánh này chỉ dùng cấu hình mặc định của cả hai mô hình (không regularization mạnh cho LR, không giới hạn độ sâu cho DT). Nếu áp dụng các biện pháp phòng thủ đã thử ở `resolve-problems.ipynb` (ví dụ regularization mạnh cho LR đã cải thiện ~5 điểm % ở mức 45% flip), khoảng cách bền vững giữa hai mô hình có thể còn thay đổi — Decision Tree cũng có công cụ tương ứng để giảm overfitting (giới hạn `max_depth`, `min_samples_leaf`) chưa được thử nghiệm ở đây.

In [ ]:
plt.plot(acc_results_df["percent"], acc_results_df["acc_lr"], marker="o", label="Logistic Regression")
plt.plot(acc_results_df["percent"], acc_results_df["acc_dt"], marker="o", label="Decision Tree")

plt.xlabel("% flip")
plt.ylabel("Accuracy")
plt.title("Comparing Robustness Against Label Flipping: Logistic Regression vs. Decision Tree")
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.05)
plt.show()

## Kết luận
 
Trong điều kiện huấn luyện mặc định (không áp dụng biện pháp phòng thủ nào), **Logistic Regression bền vững hơn Decision Tree** trước tấn công label flipping trên bộ dữ liệu này — accuracy của LR luôn cao hơn hoặc bằng DT ở mọi mức % flip đã thử, với khoảng cách rõ nhất (~7–8 điểm %) ở vùng đầu độc trung bình (15–35%). Ở cả hai đầu của phổ (0% và 45%), khoảng cách thu hẹp lại — ở dữ liệu gần sạch, cả hai đều học tốt; ở mức đầu độc rất nặng, cả hai đều suy giảm về gần mức ngẫu nhiên nên chênh lệch cũng ít có ý nghĩa.
 
Kết quả này bổ sung thêm một góc nhìn cho Case 3 (`anwareness-attack.ipynb`, đo disagreement rate giữa LR–DT tăng từ ~8% lên ~41%): giờ ta biết rõ hơn **vì sao** hai mô hình ngày càng bất đồng khi % flip tăng — không chỉ đơn thuần là "học khác nhau", mà một trong hai (Decision Tree) đang suy giảm chất lượng nhanh hơn mô hình còn lại một cách có hệ thống.